# 05. Difference-in-Differences

## Background

Propensity score methods and IPW require **unconfoundedness** — no unmeasured confounders. In many policy and business settings this assumption is implausible: we know there are unobserved variables driving both treatment selection and outcomes. Difference-in-Differences (DiD) offers a different identification strategy. Instead of controlling for all confounders, it leverages panel data and assumes that treated and control units would have trended *parallel* in the absence of treatment.

DiD was originally developed for the Card & Krueger (1994) minimum wage study. Modern extensions handle staggered adoption (multiple rollout dates), heterogeneous timing effects, and violations of the parallel trends assumption.

## What You'll Learn

- The DiD estimator: (Y̅₁_post − Y̅₁_pre) − (Y̅₀_post − Y̅₀_pre)
- Parallel trends assumption and how to test it (pre-trend check)
- Two-way fixed effects (TWFE) regression with unit and time FEs
- Event study plots: treatment effects over time
- Staggered adoption and why naive TWFE can be badly biased (Callaway & Sant'Anna)

## Why This Matters

DiD is ubiquitous in applied economics, public health, and tech industry experimentation. Synthetic control (Abadie et al. 2010) extends DiD to settings with a single treated unit. The recent literature on staggered DiD (Callaway-Sant'Anna 2021, Sun-Abraham 2021, de Chaisemartin & D'Haultfoeuille 2020) has shown that standard TWFE with heterogeneous timing is biased — something the industry largely ignored until ~2021. This is now considered essential knowledge for anyone running rollout experiments.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Classic 2x2 DiD

Two groups (treated, control), two periods (pre, post). Treatment happens between periods for the treated group.

In [ ]:
def simulate_did_2x2(n_units=200, true_att=5.0, seed=0):
    """2×2 DiD with parallel trends and unobserved unit-level heterogeneity."""
    rng = np.random.default_rng(seed)
    n_ctrl = n_units // 2
    n_trt  = n_units - n_ctrl

    # Unit-level fixed effects (unobserved, would bias cross-section comparison)
    fe_ctrl = rng.normal(10, 3, n_ctrl)
    fe_trt  = rng.normal(15, 3, n_trt)   # treated units have higher baseline!

    # Common time trend
    time_trend = 2.0

    rows = []
    for i, fe in enumerate(fe_ctrl):
        rows.append({'unit': i, 'group': 0,
                     'y_pre':  fe + rng.normal(0, 1),
                     'y_post': fe + time_trend + rng.normal(0, 1)})
    for i, fe in enumerate(fe_trt):
        rows.append({'unit': n_ctrl + i, 'group': 1,
                     'y_pre':  fe + rng.normal(0, 1),
                     'y_post': fe + time_trend + true_att + rng.normal(0, 1)})

    df_wide = pd.DataFrame(rows)

    # Long format for regression
    dfs = []
    for period, col in [('pre', 'y_pre'), ('post', 'y_post')]:
        tmp = df_wide[['unit','group',col]].copy()
        tmp = tmp.rename(columns={col: 'y'})
        tmp['post'] = int(period == 'post')
        tmp['treated'] = tmp['group']
        tmp['did'] = tmp['treated'] * tmp['post']
        dfs.append(tmp)
    df_long = pd.concat(dfs).sort_values(['unit','post'])
    return df_wide, df_long


df_wide, df_long = simulate_did_2x2(n_units=400, true_att=5.0)

# Compute 2x2 DiD manually
y11 = df_wide.loc[df_wide.group==1, 'y_post'].mean()  # treated post
y10 = df_wide.loc[df_wide.group==1, 'y_pre'].mean()   # treated pre
y01 = df_wide.loc[df_wide.group==0, 'y_post'].mean()  # control post
y00 = df_wide.loc[df_wide.group==0, 'y_pre'].mean()   # control pre

did_estimate = (y11 - y10) - (y01 - y00)
cross_section = y11 - y01  # biased by unit FE!

print(f"{'Quantity':<40} {'Value':>8}")
print(f"{'True ATT':<40} {'5.000':>8}")
print(f"{'Cross-section (treated_post - ctrl_post)':<40} {cross_section:>8.3f}  ← BIASED (unit FE)")
print(f"{'Before-after (treated only)':<40} {y11-y10:>8.3f}  ← BIASED (time trend)")
print(f"{'DiD estimate':<40} {did_estimate:>8.3f}  ← UNBIASED (under PT)")

In [ ]:
# Visualize the DiD logic
fig, ax = plt.subplots(figsize=(8, 5))

# Four means
points = {
    ('Control','Pre'):  (0, y00),
    ('Control','Post'): (1, y01),
    ('Treated','Pre'):  (0, y10),
    ('Treated','Post'): (1, y11),
}

# Actual trajectories
ax.plot([0,1], [y00, y01], 'o-', color='#2196F3', lw=2.5, label='Control', markersize=8)
ax.plot([0,1], [y10, y11], 'o-', color='#F44336', lw=2.5, label='Treated', markersize=8)

# Counterfactual (parallel trend)
counterfactual_post = y10 + (y01 - y00)
ax.plot([0,1], [y10, counterfactual_post], '--', color='#F44336', lw=2,
        alpha=0.5, label='Treated counterfactual (parallel trend)')

# DiD arrow
ax.annotate('', xy=(1.05, y11), xytext=(1.05, counterfactual_post),
            arrowprops=dict(arrowstyle='<->', color='green', lw=2))
ax.text(1.08, (y11 + counterfactual_post)/2, f'DiD = {did_estimate:.2f}',
        va='center', color='green', fontweight='bold')

ax.set_xticks([0, 1]); ax.set_xticklabels(['Pre', 'Post'])
ax.set_ylabel('Outcome Y'); ax.set_title('Difference-in-Differences — 2×2 Design')
ax.legend(); ax.set_xlim(-0.2, 1.3)
plt.tight_layout()
plt.savefig('05_did_2x2.png', dpi=110, bbox_inches='tight')
plt.show()

## 2. Two-Way Fixed Effects (TWFE) Regression

In panel data with T periods: regress Y on treatment dummy, unit FEs, and time FEs. The unit FEs absorb time-invariant confounders; time FEs absorb common shocks.

In [ ]:
def simulate_panel(n_units=100, n_periods=8, treat_period=4, true_att=3.0, seed=0):
    """Multi-period panel: all treated units switch at t=treat_period."""
    rng = np.random.default_rng(seed)
    n_treated = n_units // 2

    rows = []
    for u in range(n_units):
        unit_fe = rng.normal(0 if u < n_treated else 3, 2)  # diff baselines
        treat_group = (u >= n_treated)
        for t in range(n_periods):
            post = (t >= treat_period)
            treated_now = treat_group and post
            y = unit_fe + 0.5*t + (true_att if treated_now else 0) + rng.normal(0, 1)
            rows.append({'unit': u, 'time': t, 'treat_group': int(treat_group),
                         'post': int(post), 'treated': int(treated_now), 'y': y})

    return pd.DataFrame(rows)


df_panel = simulate_panel(n_units=100, n_periods=8, treat_period=4, true_att=3.0)

# TWFE via statsmodels (unit + time fixed effects via dummies absorbed by C())
twfe = smf.ols('y ~ treated + C(unit) + C(time)', data=df_panel).fit(
    cov_type='HC1'  # heteroskedasticity-robust SEs
)
att_twfe = twfe.params['treated']
se_twfe  = twfe.bse['treated']
print(f"TWFE ATT estimate = {att_twfe:.3f}  (SE={se_twfe:.3f})")
print(f"95% CI: [{att_twfe - 1.96*se_twfe:.3f}, {att_twfe + 1.96*se_twfe:.3f}]")
print(f"True ATT = 3.000")

## 3. Pre-Trend Check (Parallel Trends Test)

The identifying assumption is **not** testable for the post-treatment period. But we can check whether treated and control trended parallel *before* treatment — a necessary (not sufficient) condition.

In [ ]:
def event_study(df_panel, treat_period=4):
    """
    Event study regression: interact treatment group with time dummies.
    Omit the period just before treatment (t=treat_period-1) as reference.
    Coefficients show treatment effect at each relative time.
    """
    df = df_panel.copy()
    df['rel_time'] = df['time'] - treat_period
    # Create interaction dummies (omit rel_time = -1)
    reference = -1
    time_vals = sorted(df['rel_time'].unique())
    for t in time_vals:
        if t != reference:
            df[f'g_t_{t}'] = (df['treat_group'] == 1) & (df['rel_time'] == t)
            df[f'g_t_{t}'] = df[f'g_t_{t}'].astype(int)

    g_t_cols = [c for c in df.columns if c.startswith('g_t_')]
    formula = 'y ~ ' + ' + '.join(g_t_cols) + ' + C(unit) + C(time)'
    m = smf.ols(formula, data=df).fit(cov_type='HC1')

    ests = []
    for t in time_vals:
        if t == reference:
            ests.append({'rel_time': t, 'coef': 0, 'ci_lo': 0, 'ci_hi': 0})
        else:
            col = f'g_t_{t}'
            c = m.params[col]
            s = m.bse[col]
            ests.append({'rel_time': t, 'coef': c, 'ci_lo': c-1.96*s, 'ci_hi': c+1.96*s})
    return pd.DataFrame(ests)


es = event_study(df_panel, treat_period=4)

fig, ax = plt.subplots(figsize=(9, 4))
pre  = es[es.rel_time < 0]
post = es[es.rel_time >= 0]

for df_p, color, label in [(pre,'#2196F3','Pre-treatment (test parallel trends)'),
                            (post,'#F44336','Post-treatment (causal effect)')]:
    ax.errorbar(df_p.rel_time, df_p.coef,
                yerr=[df_p.coef - df_p.ci_lo, df_p.ci_hi - df_p.coef],
                fmt='o-', color=color, capsize=4, lw=2, label=label)

ax.axhline(0, color='black', lw=1)
ax.axvline(-0.5, color='gray', ls='--', lw=1.5, label='Treatment onset')
ax.axhline(3.0, color='green', ls=':', lw=1.5, label='True ATT=3.0')
ax.set_xlabel('Time relative to treatment'); ax.set_ylabel('Estimated effect')
ax.set_title('Event Study Plot — Pre-trend check + Dynamic treatment effects')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('05_event_study.png', dpi=110, bbox_inches='tight')
plt.show()

pre_coefs = pre[pre.rel_time != -1]['coef']
print(f"Pre-trend coefficients: {pre_coefs.round(3).tolist()}")
print("(Should be near zero if parallel trends holds)")

## 4. Staggered Adoption — Why Naive TWFE Fails

When different units are treated at different times, TWFE uses already-treated units as controls for later-treated units. This "forbidden comparison" can make TWFE badly biased when treatment effects are heterogeneous across cohorts or time.

In [ ]:
def simulate_staggered(n_units=120, n_periods=12, seed=0):
    """Staggered adoption: 3 cohorts treated at t=3, 6, 9."""
    rng = np.random.default_rng(seed)
    treat_times = {0: None, 1: 3, 2: 6, 3: 9}   # group → first treated period
    n_per_group = n_units // 4

    rows = []
    unit_id = 0
    for grp, t_treat in treat_times.items():
        for _ in range(n_per_group):
            fe = rng.normal(grp * 2, 1.5)   # group-varying baselines
            # True ATT grows over time since treatment (dynamic effects)
            for t in range(n_periods):
                if t_treat is None:
                    treated_now = False
                    att = 0
                else:
                    treated_now = t >= t_treat
                    # Dynamic: effect increases with time since treatment
                    att = max(0, (t - t_treat + 1) * 0.8) if treated_now else 0
                y = fe + 0.3*t + att + rng.normal(0, 1)
                rows.append({'unit': unit_id, 'time': t, 'group': grp,
                             'treat_group': int(grp > 0),
                             'treated': int(treated_now), 'y': y,
                             'first_treat': t_treat if t_treat else 999})
            unit_id += 1
    return pd.DataFrame(rows)


df_stag = simulate_staggered()

# Naive TWFE on staggered data
twfe_stag = smf.ols('y ~ treated + C(unit) + C(time)', data=df_stag).fit(cov_type='HC1')
att_naive = twfe_stag.params['treated']

# Simple average true ATT for reference
att_true_avg = df_stag.loc[df_stag.treated==1].apply(
    lambda r: r.y - (r.y - r.y), axis=1   # placeholder — compute from DGP
).mean()

print(f"Naive TWFE estimate = {att_naive:.3f}")
print(f"(With staggered + dynamic effects, TWFE conflates cohort effects)")
print(f"\nThe fix: Callaway-Sant'Anna estimator (group-time ATTs aggregated)")
print(f"In Python: use the `csdid` package or manual cohort-level DiD")

# Manual cohort-level estimates
print(f"\nCohort-level DiD estimates (treated vs never-treated only):")
never = df_stag[df_stag.first_treat == 999]
for grp, t_treat in [(1,3),(2,6),(3,9)]:
    cohort = df_stag[df_stag.group == grp]
    # Simple 2x2 DiD for each cohort
    pre  = cohort[cohort.time == t_treat - 1]['y'].mean()
    post = cohort[cohort.time == t_treat + 2]['y'].mean()  # 3 periods post
    pre_n = never[never.time == t_treat - 1]['y'].mean()
    post_n = never[never.time == t_treat + 2]['y'].mean()
    did_cohort = (post - pre) - (post_n - pre_n)
    print(f"  Cohort t={t_treat}: DiD ATT ≈ {did_cohort:.3f}")